In [ ]:
import pandas as pd
import requests
import json
from dotenv import load_dotenv
import os
import pandas as pd
import time
from time import sleep
load_dotenv()
API_KEY = os.getenv("API_KEY")

### Filtering Authors by work count

In [2]:
df1 = pd.read_csv("authors_data.csv")

indexes = df1.index[
    (df1["works_count"] >= 5) &
    (df1["works_count"] <= 5000)
].tolist()

subset = df1.loc[indexes, "author_id"]
print(subset)

0       https://openalex.org/A5022704573
1       https://openalex.org/A5079294298
2       https://openalex.org/A5036764962
3       https://openalex.org/A5038833789
4       https://openalex.org/A5090577017
                      ...               
1491    https://openalex.org/A5064837504
1492    https://openalex.org/A5006964660
1494    https://openalex.org/A5029945844
1495    https://openalex.org/A5071769816
1496    https://openalex.org/A5003859694
Name: author_id, Length: 1333, dtype: object


### Calling API for all works of all authors from Week 2

In [ ]:
import requests
import pandas as pd
from itertools import islice
from time import sleep

css_concepts = ["C157449823", "C77805123", "C162324750", "C17744445"]
quant_concepts = ["C33923547", "C121332964", "C41008148"]

concept_filter1 = f"concept.id:{'|'.join(css_concepts)}"
concept_filter2 = f"concept.id:{'|'.join(quant_concepts)}"

BASE_URL = "https://api.openalex.org/works"

def chunked(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

rows2 = []
rows3 = []
seen_work_ids = set()

for batch_num, author_batch in enumerate(chunked(subset, 25), start=1):

    author_filter = "author.id:" + "|".join(author_batch)

    filter_string = (
        f"{author_filter},"
        f"cited_by_count:>10,"
        f"authors_count:<10,"
        f"{concept_filter1},"
        f"{concept_filter2}"
    )

    cursor = "*"
    print(f"\nProcessing batch {batch_num}")

    while cursor:

        params = {
            "filter": filter_string,
            "select": "id,publication_year,cited_by_count,title,abstract_inverted_index,authorships",
            "per-page": 200,
            "cursor": cursor,
            "api_key": API_KEY
        }

        response = requests.get(BASE_URL, params=params)
        response.raise_for_status()
        data = response.json()

        results = data.get("results", [])
        if not results:
            break

        print(f"  Fetched {len(results)} works")

        for item in results:
            work_id = item.get("id")

            if work_id in seen_work_ids:
                continue
            seen_work_ids.add(work_id)

            authorships = item.get("authorships", [])
            author_ids = [
                auth
                for auth in authorships
                if auth.get("author")
            ]

            rows2.append({
                "id": work_id,
                "publication_year": item.get("publication_year"),
                "cited_by_count": item.get("cited_by_count"),
                "author_ids": [
                    auth["author"]["id"]
                    for auth in item.get("authorships", [])
                    if auth.get("author") and auth["author"].get("id")
                ],
                })

            rows3.append({
                "id": work_id,
                "title": item.get("title"),
                "abstract_index": item.get("abstract_inverted_index"),
            })

        cursor = data["meta"].get("next_cursor")
        sleep(0.1)

    sleep(0.5)

df2 = pd.DataFrame(rows2)
df3 = pd.DataFrame(rows3)

print("\nComplete!")
print(len(df2), len(df3))


Processing batch 1
  Fetched 200 works
  Fetched 137 works

Processing batch 2
  Fetched 200 works
  Fetched 164 works

Processing batch 3
  Fetched 200 works
  Fetched 105 works

Processing batch 4
  Fetched 152 works

Processing batch 5
  Fetched 200 works
  Fetched 114 works

Processing batch 6
  Fetched 200 works
  Fetched 154 works

Processing batch 7
  Fetched 200 works
  Fetched 19 works

Processing batch 8
  Fetched 200 works
  Fetched 200 works
  Fetched 51 works

Processing batch 9
  Fetched 200 works
  Fetched 132 works

Processing batch 10
  Fetched 171 works

Processing batch 11
  Fetched 200 works
  Fetched 29 works

Processing batch 12
  Fetched 139 works

Processing batch 13
  Fetched 184 works

Processing batch 14
  Fetched 151 works

Processing batch 15
  Fetched 133 works

Processing batch 16
  Fetched 200 works
  Fetched 29 works

Processing batch 17
  Fetched 157 works

Processing batch 18
  Fetched 200 works
  Fetched 180 works

Processing batch 19
  Fetched 200 

In [ ]:
df2.to_csv("works_ids.csv", index=False)
df3.to_csv("works_titles_abstracts.csv", index=False)